# NCAA Basketball Score Prediction — 2026 Tournament

**Model:** Baio & Blangiardo (2010) Bayesian hierarchical model adapted for NCAA basketball  
**Inference:** numpyro NUTS (4 chains × 1000 warmup + 1000 samples)  
**Likelihood:** NegativeBinomial2 (overdispersion vs. Poisson)  
**Data:** 2022-23 regular season — ACC, Big Ten, Big 12, SEC, Big East (~78 teams)  
**Tournament predictions:** neutral-court mode (`home_adv` zeroed out)

---
```
log θ_g1 = home_adv * (1 - is_neutral) + att[home] + def[away]
log θ_g2 =                               att[away] + def[home]
score_g1 ~ NegBin2(mean=exp(θ_g1), concentration=phi)
score_g2 ~ NegBin2(mean=exp(θ_g2), concentration=phi)
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../src/2023'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import jax
import numpyro

print('JAX version:     ', jax.__version__)
print('numpyro version: ', numpyro.__version__)
print('JAX devices:     ', jax.devices())

DATA_PATH   = '../../data/halftime_odds.tsv'
SAMPLES_PATH = 'posterior_2023.npz'
FIGS_PATH    = '../../figs/2023/'
os.makedirs(FIGS_PATH, exist_ok=True)

## 1. Data Loading & EDA

In [ ]:
from data_prep import prepare_data, CONFERENCE_TEAMS, load_raw, filter_to_conferences, build_game_table

data = prepare_data(DATA_PATH, holdout_n=30, seed=42)

train      = data['train']
holdout    = data['holdout']
team_index = data['team_index']
n_teams    = data['n_teams']
all_games  = data['all_games']

print(f'Teams:         {n_teams}')
print(f'Total games:   {len(all_games)}')
print(f'Training:      {len(train)}')
print(f'Holdout:       {len(holdout)}')
print()
print('Teams per conference:')
for conf, teams in CONFERENCE_TEAMS.items():
    in_data = [t for t in teams if t in team_index.index]
    print(f'  {conf:10s}: {len(in_data)} teams')

In [ ]:
# Score distribution EDA
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Combined scores
all_scores = pd.concat([
    train['home_score'].rename('score'),
    train['away_score'].rename('score'),
])

axes[0].hist(all_scores, bins=30, color='#003087', alpha=0.7, edgecolor='white')
axes[0].set_title('Score Distribution (all teams)')
axes[0].set_xlabel('Points')
axes[0].set_ylabel('Count')
axes[0].axvline(all_scores.mean(), color='red', linestyle='--', label=f'Mean={all_scores.mean():.1f}')
axes[0].legend()

# Spread
spread = train['home_score'] - train['away_score']
axes[1].hist(spread, bins=30, color='#CC0033', alpha=0.7, edgecolor='white')
axes[1].set_title('Spread Distribution (home − away)')
axes[1].set_xlabel('Point differential')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].axvline(spread.mean(), color='red', linestyle='--', label=f'Mean={spread.mean():.1f}')
axes[1].legend()

# Total
total = train['home_score'] + train['away_score']
axes[2].hist(total, bins=30, color='#005288', alpha=0.7, edgecolor='white')
axes[2].set_title('Total Points Distribution')
axes[2].set_xlabel('Combined points')
axes[2].axvline(total.mean(), color='red', linestyle='--', label=f'Mean={total.mean():.1f}')
axes[2].legend()

plt.tight_layout()
plt.savefig(FIGS_PATH + 'eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Score mean={all_scores.mean():.1f}  std={all_scores.std():.1f}  var/mean={all_scores.var()/all_scores.mean():.2f}')
print(f'(var/mean >> 1 justifies NegBin2 over Poisson)')

In [ ]:
# Conference game counts
conf_map = {t: c for c, ts in CONFERENCE_TEAMS.items() for t in ts}

train_with_conf = train.copy()
train_with_conf['home_conf'] = train_with_conf['home_team'].map(conf_map)
train_with_conf['away_conf'] = train_with_conf['away_team'].map(conf_map)

print('Games by conference matchup:')
matchup_counts = train_with_conf.groupby(['home_conf', 'away_conf']).size().unstack(fill_value=0)
print(matchup_counts)
print()
print('Games by location type:')
print(train['is_neutral'].value_counts().rename({0: 'home/away', 1: 'neutral'}))

## 2. Model

In [ ]:
from model import basketball_model
import numpyro, numpy as np

# Render model plate diagram (requires graphviz: pip install graphviz)
sample_args = dict(
    home_id   = train['home_id'].values[:5],
    away_id   = train['away_id'].values[:5],
    is_neutral= train['is_neutral'].values[:5],
    n_teams   = n_teams,
    home_score= train['home_score'].values[:5],
    away_score= train['away_score'].values[:5],
)

try:
    numpyro.render_model(
        basketball_model,
        model_kwargs=sample_args,
        render_distributions=True,
        filename=FIGS_PATH + 'model_plates.png',
    )
    print('Model plate diagram saved to', FIGS_PATH + 'model_plates.png')
except ImportError:
    print('graphviz not installed — skipping plate diagram (pip install graphviz)')

## 3. Inference — NUTS

Estimated runtime: ~5–10 min on M4 Pro for 4 chains × 1000 warmup + 1000 samples.  
Set `FORCE_RERUN = True` to re-run even if cached samples exist.

In [ ]:
from inference import run_nuts, save_samples, load_samples
import os

FORCE_RERUN = False

model_args = dict(
    home_id   = train['home_id'].values,
    away_id   = train['away_id'].values,
    is_neutral= train['is_neutral'].values,
    n_teams   = n_teams,
    home_score= train['home_score'].values,
    away_score= train['away_score'].values,
)

if os.path.exists(SAMPLES_PATH) and not FORCE_RERUN:
    print(f'Loading cached samples from {SAMPLES_PATH}')
    samples = load_samples(SAMPLES_PATH)
else:
    samples = run_nuts(
        basketball_model,
        model_args,
        num_warmup=1000,
        num_samples=1000,
        num_chains=4,
        chain_method='vectorized',
        seed=0,
    )
    save_samples(samples, SAMPLES_PATH)

print('\nPosterior sample shapes:')
for k, v in samples.items():
    print(f'  {k:12s}: {v.shape}')

## 4. Diagnostics

In [ ]:
from utils import plot_rhat

fig = plot_rhat(samples, save_path=FIGS_PATH + 'rhat.png')
plt.show()

In [ ]:
# Trace plots for scalar parameters
scalar_params = ['home_adv', 'phi', 'sigma_att', 'sigma_def']
fig, axes = plt.subplots(len(scalar_params), 1, figsize=(12, 3 * len(scalar_params)))

for i, param in enumerate(scalar_params):
    if param in samples:
        vals = samples[param]
        axes[i].plot(vals, alpha=0.7, linewidth=0.5)
        axes[i].set_title(f'{param}  (mean={vals.mean():.3f}, std={vals.std():.3f})')
        axes[i].set_xlabel('Sample')

plt.tight_layout()
plt.savefig(FIGS_PATH + 'trace_scalars.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Posterior summaries for scalar parameters
summary_rows = []
for param in ['home_adv', 'phi', 'sigma_att', 'sigma_def']:
    if param in samples:
        v = samples[param]
        summary_rows.append({
            'param': param,
            'mean': v.mean(),
            'std': v.std(),
            '5%': np.percentile(v, 5),
            '50%': np.percentile(v, 50),
            '95%': np.percentile(v, 95),
        })

pd.DataFrame(summary_rows).set_index('param').round(4)

## 5. Team Quality

In [ ]:
from utils import plot_team_quality

conf_map = {t: c for c, ts in CONFERENCE_TEAMS.items() for t in ts}

fig = plot_team_quality(
    samples,
    team_index,
    conferences=conf_map,
    figsize=(14, 10),
    save_path=FIGS_PATH + 'team_quality.png',
)
plt.show()

In [ ]:
# Top teams by attack - defense composite
att_mean = samples['att'].mean(axis=0)
def_mean = samples['def'].mean(axis=0)

team_names = team_index.index.tolist()
quality_df = pd.DataFrame({
    'team': team_names,
    'attack': att_mean,
    'defense': def_mean,
    'net': att_mean - def_mean,   # high attack + low defense = good
})
quality_df['conference'] = quality_df['team'].map(conf_map)
quality_df = quality_df.sort_values('net', ascending=False)

print('Top 20 teams by net strength (attack − defense):')
quality_df.head(20)[['team', 'conference', 'attack', 'defense', 'net']].round(3)

## 6. Holdout Evaluation

In [ ]:
from predict import evaluate_holdout

holdout_results = evaluate_holdout(
    basketball_model,
    samples,
    holdout,
    team_index,
)

In [ ]:
# Show holdout results
cols = ['home_team', 'away_team', 'actual_home', 'actual_away',
        'pred_home_median', 'pred_away_median', 'correct', 'p_home_wins']
holdout_results[cols].round(2)

In [ ]:
# Calibration plot: predicted spread vs actual spread
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Spread calibration
axes[0].scatter(
    holdout_results['actual_spread'],
    holdout_results['pred_home_median'] - holdout_results['pred_away_median'],
    alpha=0.6, color='#003087'
)
lims = [-40, 40]
axes[0].plot(lims, lims, 'r--', linewidth=1, label='Perfect calibration')
axes[0].set_xlabel('Actual spread')
axes[0].set_ylabel('Predicted spread (median)')
axes[0].set_title('Spread: Predicted vs Actual')
axes[0].legend()

# Total calibration
axes[1].scatter(
    holdout_results['actual_total'],
    holdout_results['pred_home_median'] + holdout_results['pred_away_median'],
    alpha=0.6, color='#CC0033'
)
tlims = [100, 180]
axes[1].plot(tlims, tlims, 'r--', linewidth=1, label='Perfect calibration')
axes[1].set_xlabel('Actual total')
axes[1].set_ylabel('Predicted total (median)')
axes[1].set_title('Total: Predicted vs Actual')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGS_PATH + 'holdout_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Tournament Predictions

All predictions use `neutral_court=True` → `home_adv` is zeroed out.

In [ ]:
from predict import predict_game, print_matchup
from utils import plot_spread_ecdf, plot_total_ecdf

print('Available teams:')
for conf, teams in CONFERENCE_TEAMS.items():
    in_data = [t for t in teams if t in team_index.index]
    print(f'  {conf}: {in_data}')

In [ ]:
# ── Example matchup 1: UConn vs Kansas ─────────────────────
result1 = predict_game(
    basketball_model, samples, team_index,
    'UConn', 'Kansas',
    neutral_court=True,
)
print_matchup(result1)

fig = plot_spread_ecdf(
    result1['spread'], 'UConn', 'Kansas',
    save_path=FIGS_PATH + 'UConn_vs_Kansas_spread.png'
)
plt.show()

fig = plot_total_ecdf(
    result1['total'], 'UConn', 'Kansas',
    save_path=FIGS_PATH + 'UConn_vs_Kansas_total.png'
)
plt.show()

In [ ]:
# ── Example matchup 2: Duke vs Purdue ──────────────────────
result2 = predict_game(
    basketball_model, samples, team_index,
    'Duke', 'Purdue',
    neutral_court=True,
)
print_matchup(result2)

fig = plot_spread_ecdf(
    result2['spread'], 'Duke', 'Purdue',
    save_path=FIGS_PATH + 'Duke_vs_Purdue_spread.png'
)
plt.show()

fig = plot_total_ecdf(
    result2['total'], 'Duke', 'Purdue',
    save_path=FIGS_PATH + 'Duke_vs_Purdue_total.png'
)
plt.show()

In [ ]:
# ── Example matchup 3: Kentucky vs Marquette ───────────────
result3 = predict_game(
    basketball_model, samples, team_index,
    'Kentucky', 'Marquette',
    neutral_court=True,
)
print_matchup(result3)

fig = plot_spread_ecdf(
    result3['spread'], 'Kentucky', 'Marquette',
    save_path=FIGS_PATH + 'Kentucky_vs_Marquette_spread.png'
)
plt.show()

In [ ]:
# ── Batch: predict all possible conference matchups ─────────
# Uncomment to run a full bracket sweep
# tournament_teams = [
#     'UConn', 'Kansas', 'Purdue', 'Duke', 'Kentucky',
#     'Marquette', 'Alabama', 'Tennessee', 'Baylor', 'Houston',
# ]
# 
# import itertools
# bracket_results = []
# for a, b in itertools.combinations(tournament_teams, 2):
#     if a in team_index.index and b in team_index.index:
#         r = predict_game(basketball_model, samples, team_index, a, b)
#         bracket_results.append({
#             'team_a': a, 'team_b': b,
#             'p_a_wins': r['p_a_wins'],
#             'spread_median': float(np.median(r['spread'])),
#             'total_median': float(np.median(r['total'])),
#         })
# pd.DataFrame(bracket_results)

## Helper: Custom Matchup

Use the cell below to predict any matchup between teams in the dataset.

In [ ]:
# ── Change these team names to predict any matchup ─────────
TEAM_A = 'UConn'
TEAM_B = 'Alabama'

result = predict_game(
    basketball_model, samples, team_index,
    TEAM_A, TEAM_B,
    neutral_court=True,
)
print_matchup(result)

fig = plot_spread_ecdf(result['spread'], TEAM_A, TEAM_B)
plt.show()
fig = plot_total_ecdf(result['total'], TEAM_A, TEAM_B)
plt.show()